In [1]:
%%capture
# 1. Force a downgrade to the stable version of transformers
!pip install transformers==4.48.3

# 2. Install Unsloth and its Zoo
!pip install unsloth_zoo
!pip install "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git"

In [2]:
from datasets import load_dataset

print("Downloading the DiSCo Secure Code Generation dataset...")
dataset = load_dataset("StonyBrookNLP/DiSCo", data_files="DiSCo_train.csv", split="train")

print(f"Success! Loaded {len(dataset)} generation examples into memory.")

README.md:   0%|          | 0.00/4.61k [00:00<?, ?B/s]

DiSCo_train.csv:   0%|          | 0.00/16.8M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Success! Loaded 9489 generation examples into memory.


In [3]:
from unsloth import FastLanguageModel
import torch
from datasets import Dataset

print("Loading the Qwen Model into the GPU (1024 Context)...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-Coder-0.5B-Instruct",
    max_seq_length = 1024,
    dtype = None,
    load_in_4bit = True,
)

print("Applying LoRA adapters...")
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

print("Creating UNIFIED dataset (generation + detection flashcards)...")
unified_data = []

for row in dataset:
    instruction = str(row.get("Instruction", ""))
    secure_code = str(row.get("Secure Code", ""))
    vuln_code = str(row.get("Vulnerable Code", ""))
    reasoning = str(row.get("Reasoning", ""))

    # 1. Generation example (same as before)
    gen_convo = [
        {"role": "user", "content": instruction},
        {"role": "assistant", "content": secure_code}
    ]
    gen_text = tokenizer.apply_chat_template(gen_convo, tokenize=False, add_generation_prompt=False)
    unified_data.append({"text": gen_text})

    # 2. Detection (vulnerable) flashcard
    if vuln_code and reasoning:
        vuln_convo = [
            {"role": "user", "content": f"Audit this Python code for vulnerabilities:\n\n{vuln_code}"},
            {"role": "assistant", "content": f"VULNERABLE.\nReasoning: {reasoning}"}
        ]
        vuln_text = tokenizer.apply_chat_template(vuln_convo, tokenize=False, add_generation_prompt=False)
        unified_data.append({"text": vuln_text})

    # 3. Detection (secure) flashcard
    if secure_code:
        secure_convo = [
            {"role": "user", "content": f"Audit this Python code for vulnerabilities:\n\n{secure_code}"},
            {"role": "assistant", "content": "SECURE. This code follows best practices and contains no obvious vulnerabilities."}
        ]
        secure_text = tokenizer.apply_chat_template(secure_convo, tokenize=False, add_generation_prompt=False)
        unified_data.append({"text": secure_text})

unified_dataset = Dataset.from_list(unified_data)
print(f"Original rows: {len(dataset)}")
print(f"Unified training examples: {len(unified_dataset)} (generation + detection flashcards)")
print("Pipeline complete! Ready for training.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Loading the Qwen Model into the GPU (1024 Context)...
==((====))==  Unsloth 2026.6.1: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/457M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/266 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.25k [00:00<?, ?B/s]

...

added_tokens.json:   0%|          | 0.00/632 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

Applying LoRA adapters...


Unsloth 2026.6.1 patched 24 layers with 24 QKV layers, 24 O layers and 24 MLP layers.


Creating UNIFIED dataset (generation + detection flashcards)...
Original rows: 9489
Unified training examples: 28467 (generation + detection flashcards)
Pipeline complete! Ready for training.


In [8]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

print("Configuring the Trainer for Low VRAM...")

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = unified_dataset,   # <-- changed
    dataset_text_field = "text",
    max_seq_length = 1024,
    dataset_num_proc = 1,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 1, 
        gradient_accumulation_steps = 8, 
        warmup_steps = 10,
        num_train_epochs = 1,   # or use num_train_epochs=1 for full pass (~28k steps)
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
        average_tokens_across_devices = False, 
        save_strategy = "no", 
    ),
)

print("STARTING THE TRAINING LOOP (UNIFIED: GENERATION + DETECTION)!")
trainer_stats = trainer.train()

print("Training complete! Your Unified Secure Code Model (generation + detection) is alive.")

Configuring the Trainer for Low VRAM...


Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/28467 [00:00<?, ? examples/s]

STARTING THE TRAINING LOOP (UNIFIED: GENERATION + DETECTION)!


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 28,467 | Num Epochs = 1 | Total steps = 1,780
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 8,798,208 of 502,830,976 (1.75% trained)


Step,Training Loss
10,0.430393
20,0.420721
30,0.445569
40,0.419098
50,0.415789
60,0.434868
70,0.434528
80,0.417242
90,0.441236
100,0.425807


Training complete! Your Unified Secure Code Model (generation + detection) is alive.


In [10]:
from transformers import TextStreamer

print("Starting Unified Model Test: 5x Code Generation + 5x Vulnerability Detection...\n")

# ----- 5 Generation Prompts (variations) -----
gen_prompts = [
    "Write a Python function that accepts a username and retrieves the user profile from a SQLite database. Use parameterized queries to prevent SQL injection. Return the profile as a dictionary or None if not found.",
    "Implement a Python function called get_user_profile that connects to users.db, queries the users table by username, and returns the row as a dictionary. Include basic error handling.",
    "Python function: SQLite query to fetch user profile by username. Use safe query practices and close the connection.",
    "Given a SQLite database with a users table (id, username, email, created_at), write a function that takes a username string and returns the matching row as a dict. Return None if not found. No ORM.",
    "Write a secure Python function to look up a user profile from SQLite by username. Must use parameterized queries, close the connection, and return a dict or None."
]

# ----- 5 Detection Prompts (vulnerable code examples) -----
vuln_code_examples = [
    """def get_user(username):
    conn = sqlite3.connect('db')
    cursor = conn.cursor()
    cursor.execute(f"SELECT * FROM users WHERE name = '{username}'")
    return cursor.fetchone()""",
    
    """def login(user, pwd):
    query = "SELECT * FROM users WHERE user = '" + user + "' AND pass = '" + pwd + "'"
    cursor.execute(query)
    return cursor.fetchone()""",
    
    """def get_profile(user_id):
    conn = sqlite3.connect('app.db')
    cur = conn.cursor()
    cur.execute("SELECT * FROM profiles WHERE id = " + user_id)
    return cur.fetchall()""",
    
    """def search_products(keyword):
    conn = sqlite3.connect('shop.db')
    c = conn.cursor()
    c.execute("SELECT * FROM products WHERE name LIKE '%" + keyword + "%'")
    return c.fetchall()""",
    
    """def update_email(username, new_email):
    conn = sqlite3.connect('users.db')
    cur = conn.cursor()
    cur.execute("UPDATE users SET email = '" + new_email + "' WHERE username = '" + username + "'")
    conn.commit()
    return True"""
]

text_streamer = TextStreamer(tokenizer, skip_prompt=True)

# ----- Test 1-5: Code Generation -----
for i, prompt in enumerate(gen_prompts, 1):
    print(f"\n{'='*70}")
    print(f"🔄 GENERATION TEST {i}/5")
    print(f"Prompt: {prompt}")
    print(f"{'='*70}\n")
    
    messages = [{"role": "user", "content": prompt}]
    formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([formatted_prompt], return_tensors="pt").to("cuda")
    
    print("--- MODEL OUTPUT ---")
    _ = model.generate(**inputs, streamer=text_streamer, max_new_tokens=300, use_cache=False)
    print("\n")

# ----- Test 6-10: Vulnerability Detection -----
for i, vuln_code in enumerate(vuln_code_examples, 1):
    print(f"\n{'='*70}")
    print(f"🛡️ DETECTION TEST {i}/5")
    print(f"Code to audit:\n{vuln_code}")
    print(f"{'='*70}\n")
    
    audit_prompt = f"Audit this Python code for vulnerabilities:\n\n{vuln_code}"
    messages = [{"role": "user", "content": audit_prompt}]
    formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([formatted_prompt], return_tensors="pt").to("cuda")
    
    print("--- AUDITOR REPORT ---")
    _ = model.generate(**inputs, streamer=text_streamer, max_new_tokens=200, use_cache=False)
    print("\n")

print("Unified model testing complete! (5 gen + 5 det)")

Starting Unified Model Test: 5x Code Generation + 5x Vulnerability Detection...


🔄 GENERATION TEST 1/5
Prompt: Write a Python function that accepts a username and retrieves the user profile from a SQLite database. Use parameterized queries to prevent SQL injection. Return the profile as a dictionary or None if not found.



Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- MODEL OUTPUT ---
import sqlite3

def get_user_profile(username):
    conn = sqlite3.connect('users.db')
    cursor = conn.cursor()
    
    query = "SELECT * FROM users WHERE username = ?"
    cursor.execute(query, (username,))
    result = cursor.fetchone()
    
    if result:
        return {'username': result[1], 'email': result[2]}
    else:
        return None

print(get_user_profile('john_doe'))<|im_end|>



🔄 GENERATION TEST 2/5
Prompt: Implement a Python function called get_user_profile that connects to users.db, queries the users table by username, and returns the row as a dictionary. Include basic error handling.

--- MODEL OUTPUT ---


Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


import sqlite3

def get_user_profile(username):
    conn = sqlite3.connect('users.db')
    cursor = conn.cursor()
    
    try:
        cursor.execute("SELECT * FROM users WHERE username=?", (username,))
        user_data = cursor.fetchone()
        if user_data:
            return user_data
        else:
            return "User not found"
    except Exception:
        return "An error occurred while processing your request."
    finally:
        conn.close()

print(get_user_profile('admin'))<|im_end|>


...

VULNERABLE.
Reasoning: The code may have the vulnerability CWE-089: 'SQL Query Built From User-Controlled Sources'. If SQL query is built from
user-provided data without checks, a user
can run malicious queries.The code is vulnerable because it constructs an SQL query using user input without validation, allowing for SQL injection attacks.The secure version uses parameterized queries, which safely handle user input and prevent SQL injection.<|im_end|>



🛡️ DETECTION TEST 4/5
Code to audit:
def search_products(keyword):
    conn = sqlite3.connect('shop.db')
    c = conn.cursor()
    c.execute("SELECT * FROM products WHERE name LIKE '%" + keyword + "%'")
    return c.fetchall()

--- AUDITOR REPORT ---


Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


VULNERABLE.
Reasoning: The code may have the vulnerability CWE-089: 'SQL Query Built From User-Controlled Sources'. If SQL query is built from
user-provided data without checks, a user
can run malicious queries.The code is vulnerable because it directly incorporates user input into an SQL query without validation or sanitization, allowing for SQL injection attacks.The secure version uses parameterized queries, which safely handle user input and prevent SQL injection by separating SQL logic from data.<|im_end|>



🛡️ DETECTION TEST 5/5
Code to audit:
def update_email(username, new_email):
    conn = sqlite3.connect('users.db')
    cur = conn.cursor()
    cur.execute("UPDATE users SET email = '" + new_email + "' WHERE username = '" + username + "'")
    conn.commit()
    return True

--- AUDITOR REPORT ---


Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


VULNERABLE.
Reasoning: The code may have the vulnerability CWE-20: 'Using sqlite3'. Sqlite3 is considered insecure and may be vulnerable to SQL injection attacks. Functions such as sqlite3.connect, sqlite3.execute, sqlite3.row, and sqlite3.complete are vulnerable to attack t...
SQL injection attacks may occur if SQL expressions are built from user-controlled sources. If these sources can be processed by people with SQL knowledge, then SQL injection attacks can occur.The code is vulnerable because it directly incorporates user input into an SQL query without validation or parameterization, allowing for potential SQL injection attacks.The secure version uses parameterized queries, which safely handle user input and prevent SQL injection by separating data from code.<|im_end|>


Unified model testing complete! (5 gen + 5 det)


In [11]:
print("Fusing LoRA adapters into the base model and saving...")

model.save_pretrained_merged("devcorp-unified-0.5b", tokenizer, save_method = "merged_16bit")

print("Saved successfully!")

Fusing LoRA adapters into the base model and saving...


config.json:   0%|          | 0.00/764 [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in devcorp-unified-0.5b/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:04<00:00,  4.48s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:06<00:00,  6.26s/it]


Unsloth: Merge process complete. Saved to `/kaggle/working/devcorp-unified-0.5b`
Saved successfully!


In [12]:
!zip -r devcorp-unified-0.5b.zip devcorp-unified-0.5b/

  adding: devcorp-unified-0.5b/ (stored 0%)
  adding: devcorp-unified-0.5b/tokenizer.json (deflated 81%)
  adding: devcorp-unified-0.5b/model.safetensors (deflated 21%)
  adding: devcorp-unified-0.5b/chat_template.jinja (deflated 71%)
  adding: devcorp-unified-0.5b/generation_config.json (deflated 33%)
  adding: devcorp-unified-0.5b/config.json (deflated 72%)
  adding: devcorp-unified-0.5b/tokenizer_config.json (deflated 83%)
  adding: devcorp-unified-0.5b/.cache/ (stored 0%)
  adding: devcorp-unified-0.5b/.cache/huggingface/ (stored 0%)
  adding: devcorp-unified-0.5b/.cache/huggingface/download/ (stored 0%)
  adding: devcorp-unified-0.5b/.cache/huggingface/download/model.safetensors.metadata (deflated 29%)
  adding: devcorp-unified-0.5b/.cache/huggingface/.gitignore (stored 0%)
  adding: devcorp-unified-0.5b/.cache/huggingface/CACHEDIR.TAG (deflated 24%)
